# Quant Market-Making Assessment — Local Practice Replica

This is an **independent, unsolved practice assessment** modeled only on the public structure of the Millennium/Amplify notebook. It is not a current or proprietary assessment. The feeds are deterministic synthetic data and the simulator is local.

Work from top to bottom. Fill each `TODO`, keep the requested names and signatures, and restart the kernel before a final run. The notebook deliberately contains **112 cells**, matching the public reference notebook's length and four-part progression.

## Rules and suggested pacing

- Use ordinary Python, pandas, NumPy, and matplotlib.
- Do not change simulator source while timing an attempt.
- Treat client actions from the client's perspective; completed-trade actions are from the market maker's perspective.
- Suggested practice window: 150 minutes, followed by a separate review pass.
- Public checks give structural feedback only; they are not hidden grading logic.

In [ ]:
from pathlib import Path
import sys

# Jupyter may start in the repository root, this notebook's directory, or its parent.
_candidates = []
for _base in (Path.cwd(), *Path.cwd().parents):
    _candidates.extend((_base, _base / 'millennium-market-making-prep'))
PROJECT_ROOT = next((path for path in _candidates
                     if (path / 'AmplifyQuantTrading').is_dir()), None)
if PROJECT_ROOT is None:
    raise ModuleNotFoundError('Could not locate millennium-market-making-prep; start Jupyter from the Python workspace')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from AmplifyQuantTrading import Data, Exchange, MarketMaker, HedgeFund as hf
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd

hf.reset(seed=2026)
pd.set_option('display.max_columns', 10)
print('Local assessment environment ready')

In [ ]:
# Re-run this cell whenever you want deterministic client responses.
hf.reset(seed=2026)
TICKERS = ['AAPL', 'FB', 'AMZN', 'NFLX', 'GOOGL']
ETF_TICKER = 'FAANG'

# Challenge 1 — Two-sided price making

Build a small market-making workflow: sample requests, match reference prices, create quote objects, send quotes to a simulated client, and update a market maker's inventory. A neutral quote must place its bid 2% below and its offer 2% above the reference price.

In [ ]:
prices = Data.get_price_series('PricestoFeedserver')
price_requests = Data.get_price_requests('PriceRequeststoFeedserver')
print(f'{len(prices):,} price rows; {len(price_requests):,} requests')
print('First request:', price_requests[0])

## Task A — Create a ten-request sample

Iterate over `price_requests` and append exactly the first ten rows to `test_requests`. Preserve order and row shape: `[ticker, date, volume]`.

In [ ]:
test_requests = []

In [ ]:
# TODO: append the first ten requests without modifying price_requests.
raise NotImplementedError('Task A')

### Task A output check

Display the result. You should have ten list rows ordered by date.

In [ ]:
print(test_requests)

### Local structural check

The cell below reports whether the required outer shape is present. It does not reveal or grade your implementation.

In [ ]:
ok = (len(test_requests) == 10 and all(isinstance(row, list) and len(row) == 3 for row in test_requests))
print('Task A shape:', 'OK' if ok else 'needs work')

## Task B — Match requests to reference prices

For every sampled request, find the row in `prices` with the same ticker and date. Build `request_with_prices` as `(request, reference_price)` tuples. Preserve sample order and reject missing or duplicate matches explicitly.

In [ ]:
request_with_prices = []

In [ ]:
# TODO: match by both ticker and date, then populate request_with_prices.
# Target shape: [(['AAPL', 10000, 1000], 74.63), ...]
raise NotImplementedError('Task B')

### Task B output check

Inspect each pair and verify that the first element is the original three-item request and the second is a positive number.

In [ ]:
for item in request_with_prices:
    print(item)

### Matching check

All ten requests should be matched once. A correct key uses both ticker and date; matching only the ticker is insufficient.

In [ ]:
shape_ok = (len(request_with_prices) == len(test_requests)
            and all(isinstance(item, tuple) and len(item) == 2 for item in request_with_prices))
print('Task B shape:', 'OK' if shape_ok else 'needs work')

## Task C — Define `QuotedTrade`

Create a class with attributes `ticker`, `trade_volume`, `ref_price`, `bid_price`, `offer_price`, and `date`. Add readable `__str__` and `__repr__` methods so logs are useful.

In [ ]:
class QuotedTrade:
    def __init__(self, ticker, trade_volume, ref_price, bid_price, offer_price, date):
        # TODO: assign all six attributes and validate obvious invalid values.
        raise NotImplementedError('Task C class')

    def __str__(self):
        # TODO: return a concise human-readable description.
        raise NotImplementedError('Task C __str__')

    def __repr__(self):
        # TODO: return a constructor-like representation.
        raise NotImplementedError('Task C __repr__')

### Quote rule

For this challenge, `SPREAD = 0.02` means **2% away on each side**, not a 2% full width: `bid = ref × 0.98`, `offer = ref × 1.02`.

In [ ]:
quoted_trades = []

In [ ]:
SPREAD = 0.02

# TODO: create one QuotedTrade per matched request and append it.
raise NotImplementedError('Task C quotes')

### Task C output check

Print the quote objects. Confirm `bid_price < ref_price < offer_price` for every row.

In [ ]:
print(quoted_trades)

### Quote shape check

The next cell checks only count and required attribute names.

In [ ]:
required_quote_fields = ('ticker', 'trade_volume', 'ref_price', 'bid_price', 'offer_price', 'date')
quotes_ok = (len(quoted_trades) == len(request_with_prices)
             and all(all(hasattr(q, field) for field in required_quote_fields) for q in quoted_trades))
print('Task C shape:', 'OK' if quotes_ok else 'needs work')

## Task D — Ask the hedge-fund client

Call `hf.show(quote)` once for each quote and store each returned `HfResponse` in `hf_responses`. Its `hf_action` is lower-case `buy`, `sell`, or `refuse`, from the client's viewpoint.

In [ ]:
hf_responses = []

In [ ]:
# TODO: submit every quote exactly once and preserve response order.
raise NotImplementedError('Task D')

### Task D output check

Inspect the simulated decisions. Accepted client buys execute at your offer; accepted client sells execute at your bid.

In [ ]:
print(hf_responses)

### Response shape check

Refusals have `trade_price=None`; accepted responses have a positive execution price.

In [ ]:
valid_actions = {'buy', 'sell', 'refuse'}
responses_ok = (len(hf_responses) == len(quoted_trades)
                and all(r.hf_action in valid_actions for r in hf_responses))
print('Task D shape:', 'OK' if responses_ok else 'needs work')

## Task E — Initialize the market maker

`MarketMaker.mm()` creates zero positions for the five stocks and the FAANG ETF. It also exposes `quoted_trades`, `completed_trades`, `ETF_positions`, and `current_positions`.

In [ ]:
mm = MarketMaker.mm()
print(mm.current_positions)

## Task F — Log every quote

Pass every object in `quoted_trades` to `mm.add_quoted_trade`. Logging all quotes is part of the audit trail, whether the client later accepts or refuses.

In [ ]:
# TODO: log each quote exactly once.
raise NotImplementedError('Task F')

### Quote-log check

The log order should equal the submission order.

In [ ]:
print(mm.quoted_trades)

## Task G — Define `CompletedTrade`

Create a record with attributes `ticker`, `trade_volume`, `trade_price`, `mm_action`, `ref_price`, `bid_price`, `offer_price`, and `date`. The action is from the **market maker's** viewpoint.

In [ ]:
class CompletedTrade:
    def __init__(self, ticker, trade_volume, trade_price, mm_action,
                 ref_price, bid_price, offer_price, date):
        # TODO: assign and validate all eight fields.
        raise NotImplementedError('Task G')

## Task H — Record accepted trades and update inventory

Ignore refusals. If the client buys, the maker sells; if the client sells, the maker buys. Create a `CompletedTrade` with that opposite action and pass it to `mm.add_trade`.

In [ ]:
# TODO: translate accepted responses into maker-side completed trades.
raise NotImplementedError('Task H')

### Completed-trade output check

Inspect the execution log and current signed positions. Positive inventory is long; negative inventory is short.

In [ ]:
print(mm.completed_trades)
print(mm.current_positions)

### Accounting check

Every completed trade should correspond to one non-refused response, while every submitted quote should remain in the quote log.

In [ ]:
accepted_count = sum(r.hf_action != 'refuse' for r in hf_responses)
log_ok = len(mm.quoted_trades) == len(quoted_trades) and len(mm.completed_trades) == accepted_count
print('Task H logs:', 'OK' if log_ok else 'needs work')

## Task I — Visualize one ticker's accepted quotes

For AAPL completed trades, collect dates, bids, and offers. Keep the three lists aligned.

In [ ]:
bid_data = []
offer_data = []
quote_dates = []

# TODO: populate the three lists from mm.completed_trades.
raise NotImplementedError('Task I accepted quotes')

### Accepted-quote data check

The three lists must have equal lengths and matching order.

In [ ]:
print('bid_data:', bid_data)
print('offer_data:', offer_data)
print('quote_dates:', quote_dates)

### Collect matching reference prices

Build `ref_data` for the same accepted AAPL dates. Avoid accidentally including refused AAPL quotes.

In [ ]:
ref_data = []

# TODO: populate ref_data in exactly the same order as quote_dates.
raise NotImplementedError('Task I references')

### Reference-price check

Confirm all four plotting vectors have equal length before plotting.

In [ ]:
print('ref_data:', ref_data)
print('lengths:', len(quote_dates), len(bid_data), len(offer_data), len(ref_data))

### Plot the three price series

Use a labelled line plot with date on the x-axis and price on the y-axis. The reference line should sit between bid and offer for neutral quotes.

In [ ]:
# TODO: plot bid_data, ref_data, and offer_data against quote_dates.
raise NotImplementedError('Task I plot')

### Challenge 1 checkpoint

Before moving on, restart and run cells 0–59 with your implementations. Check that the logs contain no duplicate quotes or executions.

In [ ]:
print('quotes:', len(mm.quoted_trades), 'completed:', len(mm.completed_trades))
print('nonzero positions:', {t: p.position_volume for t, p in mm.current_positions.items() if p.position_volume})

# Challenge 2 — Inventory-aware price skewing

Process an extended request stream. A long stock position should shift both quote sides downward to encourage a client buy and discourage another client sell. A short position should shift both sides upward.

## Skewing contract

Choose and document a deterministic rule for long, short, and flat positions. Keep `bid < offer`, keep both prices positive, and do not use future requests or prices. The public reference used piecewise shifts; your implementation may be cleaner but must follow the same economic direction.

In [ ]:
# Re-importing is harmless and makes the section boundary explicit.
from AmplifyQuantTrading import Data, MarketMaker, HedgeFund as hf
hf.reset(seed=2027)

In [ ]:
# Reset response state for a reproducible timed attempt.
hf.reset(seed=2027)

## Reuse your record classes

Challenge 2 uses the `QuotedTrade` and `CompletedTrade` definitions you wrote above. If you run this section alone, execute those class cells first.

In [ ]:
required_classes_ready = all(name in globals() for name in ('QuotedTrade', 'CompletedTrade'))
print('Record classes ready:', required_classes_ready)

## Task A — Full request loop

For each request: locate the matching price, inspect current inventory, calculate a skewed two-sided quote, log it, ask the client, and record any accepted trade with the opposite maker action. Track AAPL inventory over time for later inspection.

In [ ]:
mm = MarketMaker.mm()
prices = Data.get_price_series('PricestoFeedserverExtended')
price_requests = Data.get_price_requests('PriceRequeststoFeedserverExtended')
print(len(prices), len(price_requests))

In [ ]:
print(mm.current_positions)

In [ ]:
aapl_positions = []
aapl_position_dates = []

# TODO: implement the sequential inventory-aware quoting loop.
raise NotImplementedError('Challenge 2 Task A')

In [ ]:
print('quotes logged:', len(mm.quoted_trades))
print('trades completed:', len(mm.completed_trades))
print('final positions:', mm.current_positions)

## Task B — Visualize skewed AAPL quotes

Extract AAPL date, bid, reference, and offer series from `mm.quoted_trades`. Plot them together and explain where inventory caused an asymmetric center shift.

In [ ]:
# TODO: build aligned AAPL series and plot them.
raise NotImplementedError('Challenge 2 Task B')

# Research interlude — Beta and hedging

Estimate each component stock's beta to the FAANG ETF using aligned simple returns. These estimates will be inputs to Challenge 3.

## Beta definition

For stock return series $R_i$ and ETF returns $R_m$, compute

$$\beta_i = \frac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}.$$

Use one consistent return convention, drop only the initial missing row, and verify that the ETF variance is positive.

In [ ]:
import pandas as pd
prices_long = Data.get_price_series('PricestoFeedserverEtfUpdate')

## Equal-weight ETF construction

The five components can be rebased to 100 at the first date. Their row-wise mean is a synthetic equal-weight basket. The supplied FAANG series is a tradable ETF with temporary deviations from that basket.

In [ ]:
prices_long = Data.get_price_series('PricestoFeedserverEtfUpdate')

In [ ]:
# TODO: reshape prices_long or split it by ticker, sort by date,
# and calculate aligned simple-return Series for all six assets.
raise NotImplementedError('Beta returns')

In [ ]:
betas = {}

# TODO: calculate each component beta to FAANG and populate betas.
raise NotImplementedError('Beta estimates')

# Challenge 3 — Beta-adjusted ETF hedging

Extend Challenge 2. After every accepted stock trade, calculate its beta-weighted dollar exposure and execute an opposing FAANG hedge through the local exchange.

## Hedging tools and sign convention

Create `ExchangeTrade(ticker, trade_volume, ref_price, action, date)`. `Exchange.execute` returns an executed trade; `mm.update_ETF_position` records it. A maker stock buy creates positive exposure and therefore needs an ETF sell. A maker stock sell needs an ETF buy.

In [ ]:
mm = MarketMaker.mm()
prices = Data.get_price_series('PricestoFeedserverEtfUpdate', True)
price_requests = Data.get_price_requests('PriceRequeststoFeedserverExtended')
hf.reset(seed=2028)

In [ ]:
class ExchangeTrade:
    def __init__(self, ticker, trade_volume, ref_price, action, date):
        # TODO: assign and validate the five exchange-order fields.
        raise NotImplementedError('Challenge 3 ExchangeTrade')

In [ ]:
print(prices.head())

In [ ]:
net_exposure = []
etf_position_history = []
exposure_dates = []

# TODO: run the quote/response loop, execute one beta hedge after each fill,
# and record beta-adjusted net exposure after every request.
raise NotImplementedError('Challenge 3 Task A')

In [ ]:
hedge_log_ok = len(mm.ETF_positions) <= len(mm.completed_trades)
print('Hedge log structurally plausible:', hedge_log_ok)
print('Final positions:', mm.current_positions)

## Task B — Plot beta-adjusted net exposure

Plot `net_exposure` against `exposure_dates`, add a horizontal zero line, axis labels, and a title. Discuss large residual jumps rather than assuming the hedge is perfect.

In [ ]:
# TODO: plot beta-adjusted net exposure through time.
raise NotImplementedError('Challenge 3 net-exposure plot')

In [ ]:
# TODO: plot FAANG position units through time.
raise NotImplementedError('Challenge 3 ETF-position plot')

### Risk interpretation

Beta hedging targets common-factor sensitivity, not spread risk, idiosyncratic moves, discrete-share rounding, execution costs, or model error.

In [ ]:
# TODO: calculate and print mean absolute and maximum absolute net exposure.
raise NotImplementedError('Challenge 3 risk summary')

### Challenge 3 reflection

Write three short observations: when exposure is largest, whether hedge direction is correct, and which modeling assumption most limits the result.

# Research interlude — Real versus synthetic ETF

Construct the equal-weight synthetic basket and study `real FAANG − synthetic FAANG`. A stationary-looking spread may indicate relative-value opportunities, but visual convergence alone is not proof of arbitrage.

## Research task

Split the long price records into six aligned series, rebase each component to 100, average the five rebased components, and calculate the real-minus-synthetic spread. Summarize its distribution and inspect it through time.

In [ ]:
prices_long = Data.get_price_series('PricestoFeedserverEtfUpdate')

In [ ]:
# TODO: create aligned AAPL, FB, AMZN, NFLX, GOOGL, and FAANG Series.
raise NotImplementedError('Arbitrage research reshape')

In [ ]:
# TODO: build synthetic_faang and spread = real_faang - synthetic_faang.
raise NotImplementedError('Arbitrage research spread')

In [ ]:
# TODO: print descriptive spread statistics and note possible outliers.
raise NotImplementedError('Arbitrage research summary')

## Research visualization

Use one figure for real versus synthetic prices and another for the spread. Include labels, legends, and a zero line on the spread chart.

In [ ]:
# TODO: plot real_faang and synthetic_faang together.
raise NotImplementedError('Arbitrage comparison plot')

In [ ]:
# TODO: plot the spread with a horizontal zero line.
raise NotImplementedError('Arbitrage spread plot')

# Challenge 4 — ETF/component arbitrage

Build a stateful relative-value strategy using the commission-aware hedge-fund account. Open a six-leg position only when the expected edge exceeds round-trip costs, avoid stacking uncontrolled positions, and close all legs when the spread converges or at the final date.

In [ ]:
from AmplifyQuantTrading import Data
from AmplifyArbitrageTrading import HedgeFund as ArbitrageHedgeFund

prices = Data.get_price_series('PricestoFeedserverEtfUpdate', True)

In [ ]:
hedge_fund = ArbitrageHedgeFund.hf(
    commission_percentage=0.001,
    data=prices,
)

In [ ]:
print(prices.head())

In [ ]:
# TODO: implement entry, sizing, state tracking, exit, and final liquidation.
# Use hedge_fund.execute_order(ticker, volume, action, date) for every leg.
raise NotImplementedError('Challenge 4 Task A')

In [ ]:
# TODO: calculate total realized P&L net of commissions and print ending balance.
raise NotImplementedError('Challenge 4 results')

In [ ]:
# TODO: choose one ticker and display its trade_history as a DataFrame.
raise NotImplementedError('Challenge 4 audit trail')

# Final submission checklist

- Restart the kernel and run every completed cell in order.
- Confirm all TODOs and `NotImplementedError` statements are gone.
- Check quote/trade/hedge logs for duplicates and sign errors.
- Confirm plots have labels and aligned series.
- Explain spread, inventory, beta, and commission conventions in your own words.

Keep this notebook unsolved for fresh timed attempts; make a copy before entering an answer you want to retain.